# Pricing with different agents

In [ ]:
import sys
import os
import json
import re
import logging
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
import gradio as gr

sys.path.append('..')
from agents.agent import Agent
from agents.scanner_agent import ScannerAgent
from agents.frontier_agent import FrontierAgent
from agents.messaging_agent import MessagingAgent
from agents.deals import Deal, Opportunity, DealSelection
from deal_agent_framework import DealAgentFramework

load_dotenv(override=True)
logging.getLogger().setLevel(logging.INFO)

In [ ]:
#GET API KEYS
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists")
else:
    print("Google API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists")
else:
    print("OpenRouter API Key not set")

In [ ]:
#initialize agents
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"

openai = OpenAI()
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

In [ ]:

def extract_usd_value(text: str) -> float:
    cleaned = (text or "").replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", cleaned)
    return float(match.group()) if match else 0.0


def request_estimate(client: OpenAI, model: str, product_description: str) -> float:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": f"Estimate the price in USD. Reply with only the number.\n\n{product_description}",
            }
        ],
        max_tokens=32,
    )
    raw_text = response.choices[0].message.content or ""
    return extract_usd_value(raw_text)


class GeminiValuationAgent(Agent):
    name, color = "Gemini Valuer", Agent.MAGENTA

    def __init__(self, model: str = "gemini-2.5-flash"):
        self.client = gemini
        self.model = model

    def price(self, description: str) -> float:
        return request_estimate(self.client, self.model, description)


class RouterValuationAgent(Agent):
    name, color = "Router Valuer", Agent.CYAN

    def __init__(self, model: str):
        self.client = openrouter_client
        self.model = model

    def price(self, description: str) -> float:
        return request_estimate(self.client, self.model, description)

In [ ]:
DB = "products_vectorstore"
vector_client = chromadb.PersistentClient(path=DB)
catalog_collection = vector_client.get_or_create_collection("products")


class WeightedConsensusPricer:
    def __init__(self, agents_with_weights):
        self.agents_with_weights = agents_with_weights

    def estimate(self, description: str) -> float:
        return sum(weight * agent.price(description) for agent, weight in self.agents_with_weights)


frontier_agent = FrontierAgent(catalog_collection)
gemini_agent = GeminiValuationAgent()
router_agent_1 = RouterValuationAgent("openai/gpt-oss-120b:free")
router_agent_2 = RouterValuationAgent("google/gemma-3-27b-it:free")

ensemble_pricer = WeightedConsensusPricer(
    [
        (frontier_agent, 0.6),
        (gemini_agent, 0.2),
        (router_agent_1, 0.1),
        (router_agent_2, 0.1),
    ]
)


def ensemble_price(description: str) -> float:
    return ensemble_pricer.estimate(description)

In [ ]:
class StrategyPlannerLLM(Agent):
    name, color, MIN_PROFIT_MARGIN = "Strategy (LLM)", Agent.GREEN, 50

    def __init__(self, collection):
        self.scanner = ScannerAgent()
        self.messenger = MessagingAgent()
        self.collection = collection

    def evaluate_deal(self, deal: Deal) -> Opportunity:
        estimated_value = ensemble_price(deal.product_description)
        discount = estimated_value - deal.price
        return Opportunity(deal=deal, estimate=estimated_value, discount=discount)

    def plan(self, memory=None):
        shortlist = self.scanner.scan(memory=memory or [])
        if not shortlist or not shortlist.deals:
            return None

        candidates = shortlist.deals[:5]
        analyzed = [self.evaluate_deal(deal) for deal in candidates]
        best = max(analyzed, key=lambda opportunity: opportunity.discount)

        if best.discount <= self.MIN_PROFIT_MARGIN:
            return None

        self.messenger.alert(best)
        return best

In [ ]:
framework = DealAgentFramework()
framework.planner = StrategyPlannerLLM(framework.collection)
opportunities = framework.run()
print(len(opportunities), opportunities[-1] if opportunities else None)

In [ ]:
class ToolOrchestratedPlanner(Agent):
    name, color = "Tool-Orchestrated (LLM)", Agent.GREEN

    def __init__(self):
        self.scanner = ScannerAgent()
        self.messenger = MessagingAgent()
        self.controller = OpenAI()
        self.memory = []
        self.best_opportunity = None

    def _scan_market(self):
        scan_result = self.scanner.scan(memory=self.memory)
        return scan_result.model_dump_json() if scan_result else "No deals"

    def _estimate_value(self, description: str):
        return str(ensemble_price(description))

    def _send_notification(self, description: str, deal_price: float, estimated_true_value: float, url: str):
        estimate = float(estimated_true_value)
        self.messenger.notify(description, deal_price, estimate, url)
        self.best_opportunity = Opportunity(
            deal=Deal(product_description=description, price=deal_price, url=url),
            estimate=estimate,
            discount=estimate - deal_price,
        )
        return "ok"

    def tool_schema(self):
        return [
            {"type": "function", "function": {"name": "scan_market", "description": "Get bargains", "parameters": {"type": "object", "properties": {}}}},
            {"type": "function", "function": {"name": "estimate_value", "description": "Estimate value", "parameters": {"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"]}}},
            {"type": "function", "function": {"name": "send_notification", "description": "Notify user of deal", "parameters": {"type": "object", "properties": {"description": {"type": "string"}, "deal_price": {"type": "number"}, "estimated_true_value": {"type": "number"}, "url": {"type": "string"}}, "required": ["description", "deal_price", "estimated_true_value", "url"]}}},
        ]

    def _dispatch_tools(self, assistant_message):
        tool_messages = []
        for call in assistant_message.tool_calls:
            fn_name = call.function.name
            payload = json.loads(call.function.arguments or "{}")

            if fn_name == "scan_market":
                result = self._scan_market()
            elif fn_name == "estimate_value":
                result = self._estimate_value(payload.get("description", ""))
            elif fn_name == "send_notification":
                result = self._send_notification(
                    payload.get("description", ""),
                    payload.get("deal_price", 0),
                    payload.get("estimated_true_value", 0),
                    payload.get("url", ""),
                )
            else:
                result = ""

            tool_messages.append({"role": "tool", "content": str(result), "tool_call_id": call.id})

        return tool_messages

    def plan(self, memory=None):
        self.memory = memory or []
        self.best_opportunity = None

        conversation = [
            {"role": "system", "content": "Find deals, estimate value, notify user of best deal once."},
            {"role": "user", "content": "Scan, then estimate each, then notify for the best one. Reply OK when done."},
        ]

        while True:
            response = self.controller.chat.completions.create(
                model="gpt-4o-mini",
                messages=conversation,
                tools=self.tool_schema(),
            )
            message = response.choices[0].message
            if response.choices[0].finish_reason != "tool_calls":
                break
            conversation.append(message)
            conversation.extend(self._dispatch_tools(message))

        return self.best_opportunity

In [ ]:
autonomous = ToolOrchestratedPlanner()
result = autonomous.plan([])
print(result)

In [ ]:
dashboard_framework = DealAgentFramework()
dashboard_framework.planner = StrategyPlannerLLM(dashboard_framework.collection)


def opportunities_to_rows(opportunities):
    return [
        [
            item.deal.product_description,
            item.deal.price,
            item.estimate,
            item.discount,
            item.deal.url,
        ]
        for item in (opportunities or [])
    ]


with gr.Blocks(title="Price is Right", fill_width=True) as ui:
    state = gr.State(dashboard_framework.memory or [])
    gr.Markdown("**Price Estimation**")
    grid = gr.Dataframe(headers=["Description", "Price", "Estimate", "Discount", "URL"], wrap=True, max_height=400)
    ui.load(opportunities_to_rows, state, grid)

    def run_cycle():
        opportunities = dashboard_framework.run()
        return opportunities_to_rows(opportunities), opportunities

    gr.Button("Run").click(run_cycle, outputs=[grid, state])

ui.launch(inbrowser=True)